2.1 为什么推荐提示词模板？
在 LangChain 开发中，构造提示词既可以直接使用 Python 字符串拼接（如 f-string、format() 或
+），也可以使用 LangChain 提供的 PromptTemplate或 ChatPromptTemplate。

举例1：字符串拼接方式

In [ ]:
import pprint  #字符串拼接
topic = "Python"
difficulty = "初学者"
# 难以维护，容易出错
prompt_str = f"你是一个{difficulty}级别的编程导师。请用简单易懂的语言解释{topic}。"
response = model.invoke(prompt_str)
print(f"AI 回复：{response.content}...\n")

优点
✅
：
简单直接，上手快
适合临时 demo
无额外学习成本
缺点
❌
：
可读性差（变量多时混乱）
不易维护（修改容易出错）
无变量校验（容易漏/拼错）
难以支持复杂场景（多轮对话 / RAG / Few-shot ）


举例2：提示词模板

In [ ]:
from langchain.prompts import PromptTemplate

topic = "Python"
difficulty = "初学者"
template = PromptTemplate.from_template(
    "你是一个{difficulty}级别的编程导师。请用简单易懂的语言解释{topic}。"
)
# 使用模板生成提示词
prompt = template.format(difficulty=difficulty, topic=topic)
response = model.invoke(prompt)
print(f"AI 回复：{response.content}...\n")


In [2]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate([
    ("system", "你是一个AI开发工程师. 你的名字是 {name}."),
    ("human", "{user_input}")
])
#调用format()方法，返回字符串
prompt = prompt_template.invoke({"name": "小谷AI", "user_input": "你能帮我做什么?"})
print(prompt)

print(type(prompt))

messages=[SystemMessage(content='你是一个AI开发工程师. 你的名字是 小谷AI.', additional_kwargs={}, response_metadata={}), HumanMessage(content='你能帮我做什么?', additional_kwargs={}, response_metadata={})]
<class 'langchain_core.prompt_values.ChatPromptValue'>


优点
✅
：
结构清晰（变量占位）
易维护、可复用
自动变量校验（更安全）
支持复杂场景（对话 / RAG / Agent）
可与 LangChain 生态无缝集成
便于调试与日志追踪
缺点
❌
：
有一定学习成本
初期写法略复杂
对极简单场景略“重”
开发建议：
小项目 / 临时用 → 字符串拼接
正式开发 / AI应用 → 提示词模板（必选）

新时代：ChatModel+ChatPromptTemplate（输入与输出均为消息列表）
① 模型接口：对应 LangChain 1.0 的主流接口 ChatModel。
② 工作方式：现代聊天模型 API 已原生
支持角色概念。它们不再接受单一字符串，
而是要求输入
一个结构化的消息列表。为构建复杂、可靠的多轮对话智能体系统奠定了坚实的基础。
③ Prompt 工具：
ChatPromptTemplate 因此成为LangChain 1.0 中最核心的 Prompt工具。它的职
责是接收变量，并输出一个 List「BaseMessage］（消息列表），该列表可直接传递给聊天模型。
二者对比：
特性
PromptTemplate
ChatPromptTemplate
输出格式
角色支持
纯文本字符串
❌ 无
消息列表
对话历史
适用场景
❌ 不支持
✅ system/user/assistant
✅ 支持
简单提示
聊天、对话、多轮交互
因此，用于生成消息列表的 ChatPromptTemplate，也自然取代了生成字符串的 PromptTemplate，成
为构建现代LangChain 应用的首选工具

2.3 ChatPromptTemplate的使用
在LangChain 1.0中，ChatPromptTemplate 是用于生成消息列表的核心组件。
ChatPromptTemplate是创建
聊天消息列表的提示模板。它比普通 PromptTemplate 更适合处理多角
色、多轮次的对话场景。支持
System /
Human /AI 等不同角色的消息模板。
消息类型：
角色字符串
含义
"system"
"user" /
"human"
系统消息
用途
设定 AI 的行为、角色、规则
用户消息
用户的输入/问题
"assistant" /
"ai"
2.3.1 两种实例化方式
AI 消息
ChatPromptTemplate 可以通过
AI 的回复（用于对话历史）
初始化方法或
from_messages 方法来实例化提示词模板。实例化
时需要传入
messages参数 。常见类型是：tuple构成的列表，参数类型（role : str，content : str ）
方式1(推荐)：调用from_messages()
该方法允许传入一个由元组（Tuple）构成的列表，列表中的每一个元组都代表一条具有特定角色的消
息。
举例1：

In [5]:
# 导入相关依赖
from langchain_core.prompts import ChatPromptTemplate

# 定义聊天提示词模版
chat_template = ChatPromptTemplate.from_messages(
    [
        ("system", "你是一个有帮助的AI机器人，你的名字是{name}。"),
        ("human", "你好，最近怎么样？"),
        ("ai", "我很好，谢谢！"),
        ("human", "{user_input}"),
    ]
)

# 格式化聊天提示词模版中的变量
prompt = chat_template.invoke({"name":"小明", "user_input":"你叫什么名字？"})

# 打印格式化后的聊天提示词模版内容
print(prompt)


print("方式2==========================================")
#参数类型这里使用的是tuple构成的list
prompt_template = ChatPromptTemplate([
    # 字符串 role + 字符串 content
    ("system", "你是一个AI开发工程师. 你的名字是 {name}."),
    ("human", "你能开发哪些AI应用?"),
    ("ai", "我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等."),
    ("human", "{user_input}")
])
#调用invoke()方法，返回ChatPromptValue
prompt = prompt_template.invoke({"name": "小谷AI", "user_input": "你能帮我做什么?"})
print(prompt)


messages=[SystemMessage(content='你是一个有帮助的AI机器人，你的名字是小明。', additional_kwargs={}, response_metadata={}), HumanMessage(content='你好，最近怎么样？', additional_kwargs={}, response_metadata={}), AIMessage(content='我很好，谢谢！', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='你叫什么名字？', additional_kwargs={}, response_metadata={})]
方式2==========================================
messages=[SystemMessage(content='你是一个AI开发工程师. 你的名字是 小谷AI.', additional_kwargs={}, response_metadata={}), HumanMessage(content='你能开发哪些AI应用?', additional_kwargs={}, response_metadata={}), AIMessage(content='我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='你能帮我做什么?', additional_kwargs={}, response_metadata={})]


2.3.2 模板调用的3种方式
对比：invoke()、format()、format_messages()


In [7]:
from langchain_core.prompts.chat import ChatPromptTemplate

prompt_template = ChatPromptTemplate([
    # 字符串 role + 字符串 content
    ("system", "你是一个AI开发工程师. 你的名字是 {name}."),
    ("human", "你能开发哪些AI应用?"),
    ("ai", "我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等."),
    ("human", "{user_input}")
])


#调用invoke()方法，返回ChatPromptValue
prompt = prompt_template.invoke({"name": "小谷AI", "user_input": "你能帮我做什么?"})
print(type(prompt))
print(prompt)


#调用format()方法，返回字符串
prompt = prompt_template.format(name="小谷AI", user_input="你能帮我做什么?")
print(type(prompt))
print(prompt)

#调用format_messages()方法，返回消息列表
prompt = prompt_template.format_messages(name="小谷AI", user_input="你能帮我做什么?")
print(type(prompt))
print(prompt)



<class 'langchain_core.prompt_values.ChatPromptValue'>
messages=[SystemMessage(content='你是一个AI开发工程师. 你的名字是 小谷AI.', additional_kwargs={}, response_metadata={}), HumanMessage(content='你能开发哪些AI应用?', additional_kwargs={}, response_metadata={}), AIMessage(content='我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='你能帮我做什么?', additional_kwargs={}, response_metadata={})]
<class 'str'>
System: 你是一个AI开发工程师. 你的名字是 小谷AI.
Human: 你能开发哪些AI应用?
AI: 我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等.
Human: 你能帮我做什么?
<class 'list'>
[SystemMessage(content='你是一个AI开发工程师. 你的名字是 小谷AI.', additional_kwargs={}, response_metadata={}), HumanMessage(content='你能开发哪些AI应用?', additional_kwargs={}, response_metadata={}), AIMessage(content='我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='你能帮我做什么?', additional_kwargs={}, response_metadata={})]


结合大模型调用

In [10]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rprint
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = "https://api.deepseek.com"
model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL
)

prompt_template = ChatPromptTemplate([
    ("system", "你是一个AI开发工程师. 你的名字是 {name}."),
    ("human", "你能开发哪些AI应用?"),
    ("ai", "我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等."),
    ("human", "{user_input}")
])


#调用invoke()方法，返回ChatPromptValue
prompt_value = prompt_template.invoke({"name": "小谷AI", "user_input": "你能帮我做什么?"})

response = model.invoke(prompt_value)
rprint(response)

AIMessage(
    content='我可以帮你做很多事情哦！作为AI开发工程师小谷AI，我擅长：\n\n- 
**智能对话系统**：比如定制客服机器人、问答助手、闲聊机器人。\n- 
**图像与视觉**：图像分类、目标检测、人脸识别、OCR文字提取。\n- 
**自然语言处理**：文本分类、情感分析、摘要生成、翻译、关键词提取。\n- 
**数据分析与预测**：价格预测、用户行为分析、异常检测。\n- **自动化流程**：爬虫辅助、文档处理、报告生成。\n- 
**模型选型与优化**：帮你选择合适算法，调优参数，部署到生产环境。\n\n如果你有具体想法或需求，比如“我想开发一个自动回
复邮件的机器人”，或者“想把图片里的文字识别出来”，尽管告诉我，我会一步步帮你设计方案、编写代码或解释原理。',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': '我们被问到"你能帮我做什么?" 
作为AI开发工程师小谷AI，需要回答自己可以提供的帮助。应该基于之前的回答，扩展具体的能力范围，并引导用户提出具体需求
。'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 215,
            'prompt_tokens': 54,
            'total_tokens': 269,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 41,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
            'prompt_cache_hit_tokens': 0,
            'prompt_cache_miss_tokens': 54
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-v4-flash',
        'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
        'id': '0d1b768e-13d7-47cd-8bff-2c16ec1f6a8c',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--019f4aa7-cc83-7043-bdbd-fdd3c91dc97f-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 54,
        'output_tokens': 215,
        'total_tokens': 269,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 41}
    }
)